# Feasible Region: Edge-Triangle

This notebook loads the completed first-pass edge-triangle run from `outputs/final/data`, overlays the Erdos-Renyi curve `t=e^3` and the known upper boundary `t=e^(3/2)`, and checks the fixed-edge boundary optimizer against that upper curve.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "graphon_space").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DATA = ROOT / "outputs" / "final" / "data"
FIG = ROOT / "outputs" / "final" / "figures"
DATA, FIG

In [ ]:
samples = pd.read_parquet(DATA / "samples_triangle.parquet")
boundary = pd.read_parquet(DATA / "boundary_triangle.parquet")

pd.Series({
    "sample rows": len(samples),
    "k min": int(samples["k"].min()),
    "k max": int(samples["k"].max()),
    "boundary rows": len(boundary),
    "edge min": samples["e"].min(),
    "edge max": samples["e"].max(),
    "triangle min": samples["t_triangle"].min(),
    "triangle max": samples["t_triangle"].max(),
})

In [ ]:
curve = np.linspace(0, 1, 600)

fig, ax = plt.subplots(figsize=(8, 5.5))
sc = ax.scatter(samples["e"], samples["t_triangle"], c=samples["k"], s=3, alpha=0.25, cmap="viridis", rasterized=True)
ax.plot(curve, curve**3, color="black", lw=1.6, label="ER: t=e^3")
ax.plot(curve, curve**1.5, color="#b2182b", lw=1.6, label="upper: t=e^(3/2)")
ax.set_xlabel("edge density e")
ax.set_ylabel("triangle density t")
ax.set_title("Edge-triangle sampled feasible region")
ax.legend(loc="upper left")
fig.colorbar(sc, ax=ax, label="k")
fig

In [ ]:
upper = boundary[boundary["direction"] == "max"].sort_values("target_e").copy()
lower = boundary[boundary["direction"] == "min"].sort_values("target_e").copy()
upper["abs_error_vs_upper"] = (upper["observed_t"] - upper["known_upper"]).abs()

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(lower["target_e"], lower["observed_t"], "o-", ms=3, label="estimated lower")
ax.plot(upper["target_e"], upper["observed_t"], "o-", ms=3, label="estimated upper")
ax.plot(curve, curve**1.5, "--", color="black", lw=1.4, label="known upper")
ax.set_xlabel("edge density e")
ax.set_ylabel("triangle density t")
ax.set_title("Fixed-edge boundary validation")
ax.legend()
display(pd.Series({
    "median upper-boundary abs error": upper["abs_error_vs_upper"].median(),
    "worst upper-boundary abs error": upper["abs_error_vs_upper"].max(),
}))
fig